In [ ]:
# Thí nghiệm Run 002; không chạy lại để chọn kết quả đẹp hơn.
import subprocess,sys,os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'pyarrow>=17,<27','scikit-learn>=1.5,<2','xgboost-cpu>=2.1,<4','optuna>=4.5,<6'])

In [ ]:
# Kiểm tra snapshot và checksum.
from pathlib import Path
import sys, zipfile, shutil, json, hashlib
WORK=Path('/kaggle/working/quantstream_research') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd().parent
archives=list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if len(archives)>1:
    raise ValueError('Chỉ Add Input một phiên bản QuantStream dataset; bỏ các phiên bản cũ.')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts: raise ValueError('Invalid ZIP path')
        z.extractall(WORK)
else:
    manifests=[p for p in INPUT.rglob('manifest.json') if (p.parent/'development.parquet').exists()]
    if len(manifests)!=1: raise FileNotFoundError('Add Input: QuantStream_Kaggle_Input.zip bản mới nhất.')
    source=manifests[0].parent
    shutil.copytree(source,WORK/'snapshot',dirs_exist_ok=True)
    shutil.copytree(source.parent/'code',WORK/'code',dirs_exist_ok=True)
SNAPSHOT=WORK/'snapshot'
manifest=json.loads((SNAPSHOT/'manifest.json').read_text())
if manifest.get('contains_final_holdout') is not False: raise ValueError('Development-only input required')
if not manifest.get('code_files') or not (WORK/'code/quantstream/search.py').exists():
    raise ValueError('Input cũ: tải lại ZIP mới dành cho notebook 04/05.')
for name,expected in manifest['code_files'].items():
    if hashlib.sha256((WORK/'code'/name).read_bytes()).hexdigest()!=expected:
        raise ValueError('Source checksum mismatch: '+name)
sys.path.insert(0,str(WORK/'code'))
print('Source:',manifest['source_control'])
print('Development only. Final holdout included:',manifest['contains_final_holdout'])

In [ ]:
# Chỉ dùng development.
import torch,pandas as pd
from IPython.display import display,FileLink
from quantstream.search import run_search,load_snapshot
m,dev,bars,cfg=load_snapshot(SNAPSHOT)
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print('Rows:',len(dev),'Test excluded from:',m['split']['test_start'])
display(pd.DataFrame(m['split']['folds']))

In [ ]:
# Giữ ngân sách trial và epoch đã khóa.
N_TRIALS=12
MAX_EPOCHS=8
SEARCH_TIMEOUT=7200
OUTPUT=WORK/'run-002-model-search'
try:
    result=run_search(SNAPSHOT,OUTPUT,n_trials=N_TRIALS,max_epochs=MAX_EPOCHS,
                      timeout_seconds=SEARCH_TIMEOUT,device='auto')
finally:
    # Preserve partial evidence even if a fit fails or the budget expires.
    if OUTPUT.exists():
        archive=shutil.make_archive(str(WORK.parent/'QuantStream_Research_Output'),'zip',OUTPUT)
        print('Evidence ZIP:',archive)
        display(FileLink(archive))

In [ ]:
# Objective dưới 1 mới vượt baseline zero-return.
trials=pd.read_csv(OUTPUT/'trials.csv')
display(trials.sort_values('value').head(20))
display(pd.DataFrame(result['cost_sensitivity']))
print('Selected params:',result['selection']['params'])
print('Across-seed mean/std normalized MSE:',result['selection']['mean_objective'],result['selection']['std_objective'])
print('Status:',result['deployment_status'])
print('Download QuantStream_Research_Output.zip rồi gửi lại để phân tích và commit kết quả thật.')

In [ ]:
# Giữ tất cả trial, kể cả trial không vượt baseline.
import matplotlib.pyplot as plt
valid_trials = trials.dropna(subset=['value']).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(valid_trials)), valid_trials.value, color='#0f766e')
ax.axhline(1, color='#b45309', linestyle='--', label='Zero-return baseline')
ax.set(title='Model search: all completed objectives', xlabel='Trial (CSV order)', ylabel='MSE / zero-return MSE')
ax.legend(); ax.grid(axis='y', alpha=.2); plt.tight_layout(); plt.show()